# MotorPulse Random Forest — a beginner-friendly walkthrough

This notebook starts with the cleaned feature CSV exported by `MotorPulse_Feature_Engineering.ipynb`, then builds and checks a Random Forest classifier. It is written for someone who is new to machine learning: read each explanation, run the code cell below it, and compare your output with the description.

## Before you start

1. Open `MotorPulse_Feature_Engineering.ipynb` and run its **“Save the cleaned feature table”** cell.
2. Select a Python kernel that has NumPy and scikit-learn installed.
3. Run this notebook from top to bottom. In VS Code, use **Run All** or run one code cell at a time.

> **Important data warning:** all rows in the supplied CSV are generated examples. This notebook predicts generated `scenario_label` values, not real motor faults. Cleaning the CSV does not make synthetic values into physical sensor readings. A high score here is not evidence that the model works on real motors.

## The machine-learning task in plain language

Each CSV row is one feature window for one simulated motor. We give the computer the numeric measurements/features and the generator's scenario label. It looks for patterns in the examples and then predicts a label for a new row. This is **supervised classification**: supervised means we provide example answers; classification means the answer is a named category.

In this exercise:

- **One example / row:** a single feature window.
- **Features (`X`):** the 10 numeric inputs, such as vibration RMS and temperature.
- **Target (`y`):** `scenario_label`, the synthetic category we ask the model to predict.
- **Training:** show the model training rows with their features and target labels.
- **Testing:** hide some whole assets during training, then check whether the model predicts those held-out rows.

The target is not a verified fault diagnosis. `fault_status` is `not_assessed_synthetic` in every row, so it cannot train a real fault/no-fault classifier.

## What is a Random Forest?

Imagine one decision tree asking questions such as “is vibration RMS above this value?” Each answer leads to another question until the tree chooses a class. A Random Forest builds many trees using slightly different samples and feature choices. For classification, the trees vote; the class with the most votes is the prediction.

A tree can use a value in degrees Celsius alongside a value in m/s² without scaling them to the same range. That is why this example does not standardize the features. The model only learns patterns in the examples and labels it receives; it cannot know whether a label represents a real fault.

In [31]:
# Path helps us find the CSV even if the notebook is launched from a different folder.
from pathlib import Path
import csv

# Counter will count labels; defaultdict helps us group asset IDs by motor family.
from collections import Counter, defaultdict

# NumPy stores our feature values in a numeric array that scikit-learn can use.
import numpy as np

# scikit-learn provides the benchmark, Random Forest, and evaluation measures below.
try:
    import sklearn
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
    )
except ModuleNotFoundError as error:
    if error.name is None or not error.name.startswith("sklearn"):
        raise
    raise ModuleNotFoundError(
        "scikit-learn is missing or incomplete in this notebook's Python environment. "
        "Install it with: python -m pip install scikit-learn"
    ) from error

print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)

scikit-learn: 1.9.1
NumPy: 2.5.3


## Step 1 — Load the cleaned feature file

The feature-engineering notebook writes `motorpulse_mixed_motor_cleaned_synthetic_v1.csv`. We load that prepared file; we do **not** recreate features or repeat the cleaning here.

`Path.cwd()` means “the folder Python is currently working in.” VS Code can choose different working folders, so this cell checks the current folder and its parents. If no file is found, the error tells you which earlier notebook cell to run.

In [32]:
# This is the cleaned output created by the feature-engineering notebook.
cleaned_name = "motorpulse_mixed_motor_cleaned_synthetic_v1.csv"

# Search upwards from the current working folder. The CSV is in MotorPulsePoc.
search_roots = [Path.cwd(), *Path.cwd().parents]
candidate_paths = []
for root in search_roots:
    candidate_paths.extend([root / cleaned_name, root / "MotorPulsePoc" / cleaned_name])

# Stop at the first matching file. Raise a clear error instead of silently using another dataset.
cleaned_path = next((path for path in candidate_paths if path.is_file()), None)
if cleaned_path is None:
    raise FileNotFoundError(
        f"Cannot find {cleaned_name}. Run the export cell in MotorPulse_Feature_Engineering.ipynb first."
    )

# DictReader turns each CSV row into a dictionary: row["asset_id"], row["scenario_label"], etc.
with cleaned_path.open(newline="", encoding="utf-8") as csv_file:
    reader = csv.DictReader(csv_file)
    rows = list(reader)
    columns = reader.fieldnames

print("Cleaned input:", cleaned_path)
print("Cleaned rows:", len(rows))
print("Columns:", len(columns or []))
print("Data sources:", Counter(row["data_source"] for row in rows))
print("Quality statuses:", Counter(row["quality_status"] for row in rows))
print("Fault status values:", Counter(row["fault_status"] for row in rows))

# Peek at one row so we can see what a single example looks like.
print("\nOne example row:")
print({name: rows[0][name] for name in ["asset_id", "motor_type", "scenario_label", "fault_status", "accel_vector_rms_m_s2"]})

Cleaned input: d:\GTA\ai-project\MotorPulsePoc\motorpulse_mixed_motor_cleaned_synthetic_v1.csv
Cleaned rows: 1500
Columns: 39
Data sources: Counter({'synthetic_generator_v1': 1500})
Quality statuses: Counter({'valid': 1500})
Fault status values: Counter({'not_assessed_synthetic': 1500})

One example row:
{'asset_id': 'INDUCTION-01', 'motor_type': 'induction', 'scenario_label': 'synthetic_healthy', 'fault_status': 'not_assessed_synthetic', 'accel_vector_rms_m_s2': '0.2685'}


## Step 2 — Look at the examples and labels

Before training, inspect the target classes. The original generated file also had invalid sensor-quality examples; the feature-engineering notebook removed those from this cleaned file. We therefore expect five scenario labels here, with 300 rows per label.

Balanced counts are useful for this demonstration, but they do not represent real factory failure rates. In real monitoring, confirmed faults are often rare.

In [33]:
# Count how many rows the generator created for each scenario and motor family.
scenario_counts = Counter(row["scenario_label"] for row in rows)
motor_type_counts = Counter(row["motor_type"] for row in rows)
print("Scenario-label counts:")
for label, count in sorted(scenario_counts.items()):
    print(f"  {label:38s} {count}")
print("\nMotor family row counts:", dict(sorted(motor_type_counts.items())))
print("\nAre any rows confirmed real faults?", any(row["fault_status"] == "confirmed_fault" for row in rows))

Scenario-label counts:
  synthetic_current_deviation            300
  synthetic_healthy                      300
  synthetic_operating_change             300
  synthetic_thermal_deviation            300
  synthetic_vibration_deviation          300

Motor family row counts: {'bldc': 300, 'brushed_dc': 300, 'induction': 300, 'pmsm': 300, 'stepper': 300}

Are any rows confirmed real faults? False


## Step 3 — Choose model inputs (`X`) and answer (`y`)

Think of a practice exam: `X` is the information given to the student, and `y` is the answer the student must predict. In code, uppercase `X` is a common convention for a table of inputs; lowercase `y` is a common convention for the target answers.

We use these 10 numeric features as `X`:

| Feature | What it summarizes |
|---|---|
| `accel_x_rms_m_s2`, `accel_y_rms_m_s2`, `accel_z_rms_m_s2` | RMS vibration on each axis |
| `accel_vector_rms_m_s2` | Combined three-axis vibration RMS |
| `accel_peak_to_peak_max_m_s2` | Largest axis peak-to-peak vibration range |
| `accel_crest_factor_max` | Largest peak-to-RMS ratio across axes |
| `dominant_frequency_hz` | Frequency bin with the strongest vibration magnitude |
| `temperature_c` | Temperature feature in degrees Celsius |
| `temperature_rise_c` | Temperature above the generated ambient reference |
| `current_rms_a` | RMS current feature in amperes |

`y` is `scenario_label`. Keep `asset_id` separately for the grouped test split. Never include the target, `fault_status`, IDs, timestamps, or other labels in `X`: that would leak answers or let the model memorize records.

In [34]:
# These are the numeric inputs (X). They were already prepared upstream; we do not recalculate them here.
feature_columns = [
    "accel_x_rms_m_s2",
    "accel_y_rms_m_s2",
    "accel_z_rms_m_s2",
    "accel_vector_rms_m_s2",
    "accel_peak_to_peak_max_m_s2",
    "accel_crest_factor_max",
    "dominant_frequency_hz",
    "temperature_c",
    "temperature_rise_c",
    "current_rms_a",
]
# This is the category (y) that the classifier will practice predicting.
target_column = "scenario_label"

# The cleaned output should contain complete valid feature rows.
missing_feature_rows = [
    row for row in rows
    if any(row[column].strip() == "" for column in feature_columns)
]
assert not missing_feature_rows, f"Cleaned file unexpectedly has {len(missing_feature_rows)} incomplete rows."
assert all(row["quality_status"] == "valid" for row in rows)
assert set(row["data_source"] for row in rows) == {"synthetic_generator_v1"}

# Convert text read from the CSV into numbers. Each inner list is one row/window;
# each position in that list is one feature, in feature_columns order.
X = np.array([[float(row[column]) for column in feature_columns] for row in rows], dtype=float)

# Keep one target label and one asset ID for every feature row, in the same row order.
y = np.array([row[target_column] for row in rows])
groups = np.array([row["asset_id"] for row in rows])

print("X shape (windows, features):", X.shape)
print("y shape (one target per window):", y.shape)
print("Feature names:", feature_columns)
print("First row target:", y[0])

# With this file, the expected dimensions are 1,500 examples by 10 features.
assert X.shape == (len(rows), len(feature_columns))
assert len(y) == len(rows)

X shape (windows, features): (1500, 10)
y shape (one target per window): (1500,)
Feature names: ['accel_x_rms_m_s2', 'accel_y_rms_m_s2', 'accel_z_rms_m_s2', 'accel_vector_rms_m_s2', 'accel_peak_to_peak_max_m_s2', 'accel_crest_factor_max', 'dominant_frequency_hz', 'temperature_c', 'temperature_rise_c', 'current_rms_a']
First row target: synthetic_healthy


## Step 4 — Split by motor asset, not by random row

A **training set** is the examples the model learns from. A **test set** is held back until after training, so it can give us a fairer check on examples the model did not learn from.

Several windows belong to the same asset. If we randomly split rows, windows from one motor could land in both sets and make testing look better than it really is. Instead, we hold out one complete simulated asset from every motor family. This tests other generated assets from familiar families; it does not prove performance on real motors.

### What does `test_mask` mean?

`test_mask` is a row-by-row set of True/False answers. A value of `True` means “this row belongs to one of the held-out test assets”; `False` means it does not. `train_mask = ~test_mask` flips those answers, so the same row cannot be in both sets. For example, a row from `INDUCTION-03` is test (`True`, `False`), while a row from `INDUCTION-01` is train (`False`, `True`). We apply each mask to `X`, `y`, and the asset IDs in the same order so features stay matched with their answers.

In [35]:
# First collect the asset IDs that belong to each motor family.
assets_by_type = defaultdict(set)
for row in rows:
    assets_by_type[row["motor_type"]].add(row["asset_id"])

# Reserve the last sorted asset in each family for testing.
# Sorting makes this choice repeatable when the notebook is run again.
test_assets = {sorted(asset_ids)[-1] for asset_ids in assets_by_type.values()}

# A mask is one True/False value per row. True means the row belongs in the test set.
test_mask = np.array([asset_id in test_assets for asset_id in groups])
# The tilde (~) flips booleans: True becomes False and False becomes True.
# Therefore train_mask selects every row that is NOT from a held-out test asset.
train_mask = ~test_mask

# Apply the same masks to X, y, and asset IDs so each feature row stays with its correct label.
X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
groups_train, groups_test = groups[train_mask], groups[test_mask]
assert set(groups_train).isdisjoint(set(groups_test)), "Asset leakage: train and test share an asset."

print("Test assets:", sorted(test_assets))
# np.where finds row positions where a mask is True. Show one example from each split.
first_train_row = np.where(train_mask)[0][0]
first_test_row = np.where(test_mask)[0][0]
print("Example rows: asset ID, test_mask, train_mask")
for row_position in [first_train_row, first_test_row]:
    print(groups[row_position], bool(test_mask[row_position]), bool(train_mask[row_position]))
print("Training rows / assets:", len(y_train), len(set(groups_train)))
print("Test rows / assets:", len(y_test), len(set(groups_test)))
print("Training scenarios:", Counter(y_train))
print("Test scenarios:", Counter(y_test))

Test assets: ['BLDC-09', 'BRUSHED_DC-12', 'INDUCTION-03', 'PMSM-06', 'STEPPER-15']
Example rows: asset ID, test_mask, train_mask
INDUCTION-01 False True
INDUCTION-03 True False
Training rows / assets: 1000 10
Test rows / assets: 500 5
Training scenarios: Counter({np.str_('synthetic_healthy'): 200, np.str_('synthetic_vibration_deviation'): 200, np.str_('synthetic_thermal_deviation'): 200, np.str_('synthetic_current_deviation'): 200, np.str_('synthetic_operating_change'): 200})
Test scenarios: Counter({np.str_('synthetic_healthy'): 100, np.str_('synthetic_vibration_deviation'): 100, np.str_('synthetic_thermal_deviation'): 100, np.str_('synthetic_current_deviation'): 100, np.str_('synthetic_operating_change'): 100})


## Step 5 — Create a simple baseline to beat

A **baseline** is a simple reference score. `DummyClassifier` ignores the feature values and always predicts the most common training label. Because the five generated classes are balanced, this baseline should score about 1 out of 5, or 20% accuracy.

If a complicated model cannot do better than this simple guess, it has not shown that it learned useful patterns.

In [36]:
# Create and fit the simple reference model using training rows only.
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_predictions = dummy.predict(X_test)
dummy_accuracy = accuracy_score(y_test, dummy_predictions)
print(f"Majority-class benchmark accuracy: {dummy_accuracy:.3f}")

Majority-class benchmark accuracy: 0.200


## Step 6 — Train the Random Forest

The settings below are called **hyperparameters**: choices we make before training.

- `n_estimators=200`: build 200 trees; their votes are combined.
- `min_samples_leaf=2`: require at least two training examples in a leaf, which can discourage very specific rules.
- `max_features="sqrt"`: let each split consider a subset of the available features.
- `random_state=42`: make the model's random choices repeatable.
- `n_jobs=-1`: let scikit-learn use available CPU cores.

`fit(X_train, y_train)` is the learning step. The test rows must stay hidden until evaluation. Otherwise, we would be checking the model on examples it already studied.

In [37]:
# Create the forest with the settings explained above.
forest = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)
# fit means “learn patterns from these training examples and their labels.”
forest.fit(X_train, y_train)
print("Training complete. The test data has not been used to fit the model.")

Training complete. The test data has not been used to fit the model.


## Step 7 — Evaluate on the held-out assets

We now ask the trained model to predict the held-out feature rows.

- **Accuracy:** correct predictions divided by all test rows.
- **Balanced accuracy:** average recall across classes, so each class has equal weight.
- **Precision for a class:** of the rows predicted as that class, how many truly have that synthetic label?
- **Recall for a class:** of the rows with that synthetic label, how many did the model find?
- **Confusion matrix:** each row is the true class and each column is the predicted class. Correct predictions run down the diagonal; off-diagonal counts are mistakes.

Compare Random Forest accuracy with the baseline from Step 5. In this fixed run, 0.706 means **353 of the 500 test rows** were assigned the correct generated scenario; 0.200 means the simple baseline got **100 of 500** right. All scores here evaluate generated `scenario_label` values only—not fault detection.

For the confusion matrix, first read the printed class order. The first number in each row is the number predicted as the first class, the second is the number predicted as the second class, and so on. For example, the current-deviation row begins `[42, 54, 4, 0, 0]`: 42 were correctly predicted as current-deviation, while 54 were predicted as healthy. Those are synthetic scenario mix-ups, not missed real faults.

With this fixed synthetic CSV and the settings below, an example run gives about **20%** baseline accuracy and **70.6%** Random Forest accuracy. Small differences can occur if you change the data, split, or model settings. Neither number is real fault-detection performance.

In [38]:
# predict returns one predicted scenario label for each held-out row.
predictions = forest.predict(X_test)
class_names = sorted(set(y_train) | set(y_test))

print(f"Majority benchmark accuracy: {dummy_accuracy:.3f}")
print(f"Random Forest accuracy:      {accuracy_score(y_test, predictions):.3f}")
print(f"Random Forest balanced accuracy: {balanced_accuracy_score(y_test, predictions):.3f}")
print("\nClassification report (synthetic scenarios):")
print(classification_report(y_test, predictions, labels=class_names, zero_division=0))
print("Confusion matrix (rows=true, columns=predicted):")
print("Column order (predicted):", class_names)
# Print each matrix row beside its true class name.
matrix = confusion_matrix(y_test, predictions, labels=class_names)
for true_class, matrix_row in zip(class_names, matrix):
    print(f"{true_class:38s} {matrix_row}")

Majority benchmark accuracy: 0.200
Random Forest accuracy:      0.706
Random Forest balanced accuracy: 0.706

Classification report (synthetic scenarios):
                               precision    recall  f1-score   support

  synthetic_current_deviation       0.48      0.42      0.45       100
            synthetic_healthy       0.47      0.75      0.58       100
   synthetic_operating_change       0.71      0.44      0.54       100
  synthetic_thermal_deviation       1.00      0.99      0.99       100
synthetic_vibration_deviation       1.00      0.93      0.96       100

                     accuracy                           0.71       500
                    macro avg       0.73      0.71      0.71       500
                 weighted avg       0.73      0.71      0.71       500

Confusion matrix (rows=true, columns=predicted):
Column order (predicted): [np.str_('synthetic_current_deviation'), np.str_('synthetic_healthy'), np.str_('synthetic_operating_change'), np.str_('synthetic

## Step 8 — Follow one prediction from input to answer

For one test example, we show the motor ID, the predicted scenario, and the generator's known scenario. In a real application, the true label would not be available at prediction time; it is shown here only because this is a labeled test dataset.

`predict_proba` reports the share of trees voting for each synthetic class. It is not a calibrated probability of fault, and it must not be shown as a real motor-health probability.

In [39]:
# Pick the first test row. Change 0 to another valid test-row position to inspect it.
example_index = 0
one_window = X_test[example_index].reshape(1, -1)
predicted_scenario = forest.predict(one_window)[0]
scenario_votes = forest.predict_proba(one_window)[0]
vote_index = int(np.argmax(scenario_votes))

print("Held-out asset:", groups_test[example_index])
print("Predicted synthetic scenario:", predicted_scenario)
print("Known generator scenario (for evaluation only):", y_test[example_index])
print(f"Forest vote fraction for predicted class: {scenario_votes[vote_index]:.3f}")
print("This is not a fault probability or real diagnostic.")

Held-out asset: INDUCTION-03
Predicted synthetic scenario: synthetic_healthy
Known generator scenario (for evaluation only): synthetic_healthy
Forest vote fraction for predicted class: 0.804
This is not a fault probability or real diagnostic.


## Step 9 — Inspect feature importance carefully

A Random Forest can estimate how much each input helped split the training examples. A larger value means the forest relied more on that feature in its splits. It does **not** mean the feature causes a fault or is the most important sensor in a real factory. Similar features can share importance.

This ranking describes only the generated dataset, this model, and this one train/test split.

In [40]:
# argsort gives the feature positions from low to high; [::-1] reverses to high to low.
importance_order = np.argsort(forest.feature_importances_)[::-1]
for index in importance_order:
    print(f"{feature_columns[index]:34s} {forest.feature_importances_[index]:.3f}")

temperature_rise_c                 0.220
temperature_c                      0.166
current_rms_a                      0.143
accel_vector_rms_m_s2              0.126
accel_z_rms_m_s2                   0.093
accel_x_rms_m_s2                   0.067
accel_y_rms_m_s2                   0.065
dominant_frequency_hz              0.057
accel_peak_to_peak_max_m_s2        0.052
accel_crest_factor_max             0.012


## Step 10 — What did we build, and what is still missing?

**It demonstrates:** loading the cleaned feature table, making a grouped train/test split, training a Random Forest, and evaluating its predictions of synthetic scenarios.

**It does not demonstrate:** real sensor feature extraction, confirmed fault detection, motor health, remaining life, or accuracy on industrial equipment. The cleaned file still has `data_source = synthetic_generator_v1`, and its fault status is unassessed.

For a real fault model, first collect physical sensor data and inspection-confirmed labels. Use `confirmed_fault` and `confirmed_no_fault` only when supported by evidence; keep uninspected cases `unknown`. Split by real motor and later session. Compare any classifier to the existing per-motor anomaly baseline in read-only shadow operation before considering deployment.

### Try it yourself

1. In Step 6, change `n_estimators` from 200 to 20, rerun the model cell and all cells below it, and compare the scores. Does the score change? Is a smaller forest faster?
2. In Step 3, remove one feature name from `feature_columns`, then rerun from Step 3 onward. Does performance change? Do not remove the target.
3. In Step 4, reserve a different asset from each family for testing. Do the scores stay the same? A different split can produce different results.

When changing an earlier cell, rerun every later cell so the notebook state stays consistent.

### Small glossary

- **Row / example / window:** one observation in the CSV.
- **Feature:** an input value that describes the observation.
- **Target / label:** the answer the model is trained to predict.
- **Train/test split:** separate examples used for learning versus checking.
- **Prediction:** the model's guessed label for an input row.
- **Metric:** a number that summarizes model performance.

### Final checklist

- [ ] This model reads the cleaned output of the feature-engineering notebook.
- [ ] `scenario_label` is the synthetic practice target; it is not a fault label.
- [ ] `fault_status` and IDs are excluded from inputs.
- [ ] Test assets are separate from training assets.
- [ ] Scores are reported only as synthetic scenario-demo results.
- [ ] Real fault labels will come from independent inspections, not model predictions.
- [ ] A real pilot will use later sessions and whole real assets for testing, then run in shadow mode before any operational use.